# 02 — SAMSum Preprocessing and Byte-Level BPE

Load the SAMSum splits, inspect examples, train a byte-level BPE tokenizer on the **training split only**, and encode prompt-plus-summary sequences.

Expected raw files: `data/raw/train.json`, `data/raw/val.json`, `data/raw/test.json`. Each JSON file should contain a list of records with `dialogue` and `summary` keys.

In [ ]:
from pathlib import Path
import json, random, numpy as np
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors

ROOT = Path.cwd()
if not (ROOT / "configs/config.json").exists():
    ROOT = Path("/kaggle/working/SAMSum-GPT-TensorFlow")
with open(ROOT / "configs/config.json") as f:
    CFG = json.load(f)

SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED)
RAW = ROOT / "data/raw"
PROC = ROOT / "data/processed"
TOK_DIR = ROOT / "artifacts/tokenizer"
PROC.mkdir(parents=True, exist_ok=True); TOK_DIR.mkdir(parents=True, exist_ok=True)

def load_split(filename):
    path = RAW / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Add the SAMSum {filename} file to data/raw/.")
    data = json.loads(path.read_text())
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    rows = []
    for item in data:
        dialogue, summary = item.get("dialogue"), item.get("summary")
        if dialogue and summary:
            rows.append({"dialogue": str(dialogue).strip(), "summary": str(summary).strip()})
    return rows

train_rows = load_split("train.json")
val_rows = load_split("val.json")
test_rows = load_split("test.json")
print("Train / validation / test:", len(train_rows), len(val_rows), len(test_rows))
print("\nExample dialogue:\n", train_rows[0]["dialogue"])
print("\nReference summary:\n", train_rows[0]["summary"])

In [ ]:
# Train a byte-level BPE tokenizer using training text only.
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>", "<dialogue>", "<summary>"]
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=CFG["tokenizer"]["vocab_size"],
    min_frequency=CFG["tokenizer"]["min_frequency"],
    special_tokens=special_tokens,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)
def training_iterator():
    for row in train_rows:
        yield row["dialogue"]
        yield row["summary"]

tokenizer.train_from_iterator(training_iterator(), trainer=trainer)
tokenizer.post_processor = processors.TemplateProcessing(
    single="$A",
    special_tokens=[]
)
tokenizer_path = TOK_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_path))
print("Tokenizer saved:", tokenizer_path)
print("Actual vocabulary size:", tokenizer.get_vocab_size())
print("Special IDs:", {t: tokenizer.token_to_id(t) for t in special_tokens})

In [ ]:
# Encode each record as prompt + target. The trainer later masks prompt labels.
PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
DIALOGUE_ID = tokenizer.token_to_id("<dialogue>")
SUMMARY_ID = tokenizer.token_to_id("<summary>")
MAX_LEN = CFG["sequence"]["max_length"]
MAX_PROMPT = CFG["sequence"]["max_prompt_length"]
MAX_SUMMARY = CFG["sequence"]["max_summary_length"]

def encode_record(row):
    prompt_text = "Dialogue:\n" + row["dialogue"] + "\nSummary:\n"
    prompt_ids = tokenizer.encode(prompt_text).ids
    summary_ids = tokenizer.encode(row["summary"]).ids
    # Reserve room for at least one target token and EOS.
    prompt_ids = prompt_ids[:MAX_PROMPT]
    summary_ids = summary_ids[:MAX_SUMMARY]
    ids = [BOS_ID] + prompt_ids + [SUMMARY_ID] + summary_ids + [EOS_ID]
    ids = ids[:MAX_LEN]
    return {"input_ids": ids, "prompt_length": min(len(prompt_ids) + 2, len(ids)-1),
            "reference": row["summary"], "dialogue": row["dialogue"]}

def save_jsonl(rows, filename):
    path = PROC / filename
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Saved", len(rows), "rows to", path)

train_encoded = [encode_record(r) for r in train_rows]
val_encoded = [encode_record(r) for r in val_rows]
test_encoded = [encode_record(r) for r in test_rows]
save_jsonl(train_encoded, "train.jsonl")
save_jsonl(val_encoded, "validation.jsonl")
save_jsonl(test_encoded, "test.jsonl")
print("First encoded length:", len(train_encoded[0]["input_ids"]))

## Important

The tokenizer is fit only on the training split to avoid vocabulary leakage. The preprocessing output includes the original dialogue and reference summary for evaluation, plus token IDs and the prompt boundary for loss masking.